# Transformer Fine-Tuning — ClauseGuard

Fine-tunes a transformer model for multi-label classification on the
**11** contract clause categories (expanded from the original 6 -- if you
already have a fine-tuned model from the 6-category version, it is no
longer compatible and needs to be retrained with this notebook).

**Setup:** Runtime → Change runtime type → GPU. Upload `clauses_dataset.csv`
when prompted.

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

CATEGORIES = [
    "Termination For Convenience",
    "Renewal Term",
    "Notice Period To Terminate Renewal",
    "Cap On Liability",
    "Uncapped Liability",
    "Non-Compete",
    "Audit Rights",
    "Insurance",
    "Exclusivity",
    "Warranty Duration",
    "Volume Restriction",
]

df = pd.read_csv("clauses_dataset.csv")
print(f"Loaded {len(df)} clauses across {len(CATEGORIES)} categories")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
NUM_LABELS = len(CATEGORIES)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

texts = df["clause_text"].tolist()
labels = df[CATEGORIES].values.astype(np.float32)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ClauseDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LENGTH)
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ClauseDataset(train_texts, train_labels)
test_dataset = ClauseDataset(test_texts, test_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=20,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()
print(f"Macro F1: {results['eval_macro_f1']:.3f}")
print(f"Micro F1: {results['eval_micro_f1']:.3f}")

In [ ]:
from sklearn.metrics import classification_report

preds_output = trainer.predict(test_dataset)
probs = 1 / (1 + np.exp(-preds_output.predictions))
preds = (probs > 0.5).astype(int)

print(classification_report(test_labels, preds, target_names=CATEGORIES, zero_division=0))

## Save and download the model

Extract the downloaded archive into `models/legalbert_finetuned/` in the
project directory.

In [ ]:
import shutil
from google.colab import files

SAVE_DIR = "legalbert_finetuned"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)
files.download(f"{SAVE_DIR}.zip")